# 07b — MoE Predict + Submit (Router: HCM Expert vs Generalist)

Dùng `u_top_city` từ lịch sử test_user để route qua đúng sub-model.
- user top_city ∈ HCM_CITY_NAMES → Model HCM Expert
- user top_city khác hoặc cold (null) → Model Generalist

Output: `cache_drive/submission_moe.csv`


In [ ]:
print('Skipping pip install (local kernel).')

In [ ]:
import os, sys, time, json
import numpy as np
import pandas as pd
import lightgbm as lgb
from hashlib import md5

PROJECT_DIR  = r"d:/Datathon_2"
TRAINING_DIR = os.path.join(PROJECT_DIR, "training")
CACHE_DIR    = os.path.join(PROJECT_DIR, "cache_drive")
os.makedirs(CACHE_DIR, exist_ok=True)
for p in (TRAINING_DIR, PROJECT_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)

from utils.features import build_user_features, build_item_features, add_cross_features
from utils.metrics import mean_recall_at_k, mean_ndcg_at_k
print("Imports OK | CACHE_DIR:", CACHE_DIR)


In [ ]:
TRAIN_DATE_END = "2026-04-09"
VALID_START    = "2026-03-13"

# HCM city names as they appear in city_name column
HCM_CITY_NAMES = {
    "Hồ Chí Minh", "Ho Chi Minh", "TP. Hồ Chí Minh", "TP.HCM",
    "TP Hồ Chí Minh", "Thành phố Hồ Chí Minh",
}

MONO_MAP = {
    "age_at_train_end": -1, "recency_evt_days": -1, "u_recency_days": -1,
    "i_CR_30d": 1, "i_contacts_24h_mean_30d": 1,
    "i_n_pos_30d": 1, "u_n_pos_30d": 1,
}
DROP_COLS = {"user_id","item_id","label","title","posted_date","expected_expired_date",
             "first_evt_date","last_evt_date","last_snap_date","project_id","_h","_geo"}

INTENT_WEIGHT = {
    "view_phone": 3.0, "contact_chat": 2.0,
    "contact_zalo": 2.0, "contact_sms": 2.0,
    "other_interaction": 1.0,
}
print("Constants OK | HCM aliases:", len(HCM_CITY_NAMES))


In [ ]:
# ---- Load candidates + features for full test set ----
# (run nb07 first to produce the test matrix OR reproduce inline)
import os

test_mat_path = f"{CACHE_DIR}/test_matrix.parquet"

t0 = time.time()
if os.path.exists(test_mat_path):
    full = pd.read_parquet(test_mat_path)
    print(f"Loaded test_matrix: {full.shape} | {time.time()-t0:.1f}s")
else:
    from utils.covis import build_covis
    from utils.candidates import (gen_history_candidates, gen_covis_candidates,
                                   gen_popularity_candidates, gen_content_candidates,
                                   merge_candidates)

    events_pos = pd.read_parquet(f"{CACHE_DIR}/events_positive.parquet")
    events_pos["event_ts"] = pd.to_datetime(events_pos["event_ts"])
    enr = pd.read_parquet(f"{CACHE_DIR}/dim_listing_enriched.parquet")
    test_users_df = pd.read_parquet(f"{CACHE_DIR}/test_users.parquet")
    snap_60d = pd.read_parquet(f"{CACHE_DIR}/snapshot_60d.parquet")
    snap_60d["date"] = pd.to_datetime(snap_60d["date"])

    TEST_UIDS = set(test_users_df["user_id"])
    events_pos_test = events_pos[events_pos["user_id"].isin(TEST_UIDS)].copy()

    allowed_a  = set(enr[enr["tier"] == "A"]["item_id"])
    allowed_ab = set(enr[enr["tier"].isin(["A","B"])]["item_id"])

    hist  = gen_history_candidates(events_pos_test, allowed_ab, top_n=30)
    covis = build_covis(events_pos[events_pos["item_id"].isin(allowed_a)][
        ["user_id","session_id","item_id","event_type","event_ts"]],
        allowed_items=allowed_a, top_k_per_item=20, time_decay=True)
    cvis  = gen_covis_candidates(events_pos_test, covis, allowed_a, top_n=200)

    def _mode(s):
        s = s.dropna(); return s.mode().iloc[0] if len(s) else None
    SEG_MAP = [("category","u_top_category"),("city_name","u_top_city"),
               ("district_name","u_top_district"),("ad_type","u_top_ad_type")]
    enr_seg = enr[[c for c in ["item_id","category","city_name","district_name","ad_type"] if c in enr.columns]]
    tp_t = events_pos_test.merge(enr_seg, on="item_id", how="left")
    prof_specs = {out: (src, _mode) for src, out in SEG_MAP if src in tp_t.columns}
    user_prof  = tp_t.groupby("user_id").agg(**prof_specs).reset_index()
    missing = TEST_UIDS - set(user_prof["user_id"])
    if missing:
        cold_row = {"user_id": list(missing)}
        for src, out in SEG_MAP:
            if src in tp_t.columns:
                m = tp_t[src].mode(); cold_row[out] = m.iloc[0] if len(m) else None
        user_prof = pd.concat([user_prof, pd.DataFrame(cold_row)], ignore_index=True)

    last14_start = pd.Timestamp(TRAIN_DATE_END) - pd.Timedelta(days=14)
    snap_14 = snap_60d[snap_60d["date"] >= last14_start]
    pop  = gen_popularity_candidates(user_prof, enr, snap_14, 50, 100)
    cont = gen_content_candidates(user_prof, enr, top_n=50)
    cands = merge_candidates({"history":hist,"covis":cvis,"pop":pop,"content":cont}, cap_total=500)

    pci_raw = pd.read_parquet(f"{CACHE_DIR}/pci_full.parquet",
                               columns=["user_id","item_id","purchased"])
    _purchased = pci_raw[pci_raw["purchased"]==True][["user_id","item_id"]].drop_duplicates().copy()
    _purchased["_drop"] = True
    cands = cands.merge(_purchased, on=["user_id","item_id"], how="left")
    cands = cands[cands["_drop"].isna()].drop(columns=["_drop"])
    del pci_raw, _purchased

    missing_uids = TEST_UIDS - set(cands["user_id"])
    if missing_uids:
        pop_global = (snap_14.groupby("item_id")["contacts_24h"].sum()
                      .sort_values(ascending=False).head(100).index.tolist())
        pop_global = [it for it in pop_global if it in allowed_ab]
        fb = [{"user_id":u,"item_id":it,"src_history":float("nan"),"src_covis":float("nan"),
               "src_pop":1.0,"src_content":float("nan")} for u in missing_uids for it in pop_global[:100]]
        cands = pd.concat([cands, pd.DataFrame(fb)], ignore_index=True)

    CUTOFF = pd.Timestamp(TRAIN_DATE_END) + pd.Timedelta(seconds=1)
    pv_path = f"{CACHE_DIR}/events_pageview_30d.parquet"
    pv = pd.read_parquet(pv_path) if os.path.exists(pv_path) else \
         pd.DataFrame(columns=["user_id","item_id","event_ts","dwell_time_sec"])
    if "event_ts" in pv.columns:
        pv["event_ts"] = pd.to_datetime(pv["event_ts"])

    # pass dim_enriched for u_price_pref_log super feature
    uf  = build_user_features(events_pos, pv, cutoff_ts=CUTOFF, dim_enriched=enr)
    itf = build_item_features(events_pos, snap_60d, enr, cutoff_ts=CUTOFF)
    full = add_cross_features(cands, uf, itf)
    full.to_parquet(test_mat_path, index=False)
    print(f"Built + saved test_matrix: {full.shape} | {time.time()-t0:.1f}s")

In [ ]:
# ---- Load both sub-models ----
with open(f"{CACHE_DIR}/model_hcm_feats.json")     as f: feats_hcm     = json.load(f)
with open(f"{CACHE_DIR}/model_general_feats.json")  as f: feats_general = json.load(f)
model_hcm     = lgb.Booster(model_file=f"{CACHE_DIR}/model_hcm.txt")
model_general = lgb.Booster(model_file=f"{CACHE_DIR}/model_general.txt")
print(f"model_hcm     feats: {len(feats_hcm)}")
print(f"model_general feats: {len(feats_general)}")


In [ ]:
# ---- Router: classify each user as HCM or OTHER based on u_top_city ----
if "u_top_city" in full.columns:
    full["_route"] = full["u_top_city"].apply(
        lambda c: "HCM" if (isinstance(c, str) and c.strip() in HCM_CITY_NAMES) else "OTHER"
    )
else:
    full["_route"] = "OTHER"

route_counts = full.groupby("user_id")["_route"].first().value_counts()
print("Router distribution (users):")
print(route_counts)


In [ ]:
# ---- Predict with the correct sub-model per user ----
def _align_and_predict(df, model, feat_list):
    """Align df columns to model's feature list, predict."""
    for c in feat_list:
        if c not in df.columns:
            df[c] = float("nan")
    X = df[feat_list].copy()
    for c in X.columns:
        if X[c].dtype == "object":
            X[c] = X[c].astype("category")
    return model.predict(X, num_iteration=model.best_iteration)

t0 = time.time()
full["_pred"] = float("nan")

mask_hcm   = full["_route"] == "HCM"
mask_other = full["_route"] == "OTHER"

if mask_hcm.any():
    full.loc[mask_hcm, "_pred"] = _align_and_predict(
        full[mask_hcm].copy(), model_hcm, feats_hcm)

if mask_other.any():
    full.loc[mask_other, "_pred"] = _align_and_predict(
        full[mask_other].copy(), model_general, feats_general)

print(f"Predicted in {time.time()-t0:.0f}s | null preds: {full['_pred'].isna().sum():,}")


In [ ]:
from utils.diversify import diversify_top_k
from utils.submit import validate_submission, write_submission

# Top-30 per user -> diversify -> top-10
top30 = (full.sort_values(["user_id","_pred"], ascending=[True,False])
         .groupby("user_id").head(30)[["user_id","item_id","_pred"]]
         .rename(columns={"_pred":"score"}))

enr = pd.read_parquet(f"{CACHE_DIR}/dim_listing_enriched.parquet",
                       columns=["item_id","seller_id","district_name"])
top10 = diversify_top_k(top30, enr, k=10,
                         max_per_seller=7, max_per_district=8,
                         freshness_boost=0.05, fresh_age_days=7)
print(f"top10 after diversify: {len(top10):,} | users: {top10['user_id'].nunique():,}")

# Validate + write
test_users_df = pd.read_parquet(f"{CACHE_DIR}/test_users.parquet")
TEST_UIDS = set(test_users_df["user_id"])
enr_full = pd.read_parquet(f"{CACHE_DIR}/dim_listing_enriched.parquet", columns=["item_id"])
valid_items = set(enr_full["item_id"])

validate_submission(top10, valid_items, TEST_UIDS, k=10)
out = f"{CACHE_DIR}/submission_moe.csv"
write_submission(top10, out)
print(f"\nMoE Submission ready: {out}")
print(f"File size: {os.path.getsize(out)/1024**2:.2f} MB")
